In [24]:
from fxpmath import Fxp
import numpy as np
import math

N = 12
FRAC = N-1
K = 6

ADDRtype = Fxp(val=None, signed=False, n_word=int(np.log2(2 ** (K-1))), n_frac=0)
DATA = Fxp(val=None, signed=True, n_word=N, n_frac=FRAC)

x = np.array([0.99951171875, 0, 0, 0, 0, 0])
a = np.array([0.0197, 0.1324, 0.3479, 0.3479, 0.1324, 0.0197])
assert(x.size == a.size)

xf = Fxp(x).like(DATA)
LUT = Fxp(np.array([-(1/2)*(a[0] + a[1] + a[2] + a[3] + a[4] + a[5]),
                   -(1/2)*(a[0] + a[1] + a[2] + a[3] + a[4] - a[5]),
                   -(1/2)*(a[0] + a[1] + a[2] + a[3] - a[4] + a[5]),
                   -(1/2)*(a[0] + a[1] + a[2] + a[3] - a[4] - a[5]),
                   -(1/2)*(a[0] + a[1] + a[2] - a[3] + a[4] + a[5]),
                   -(1/2)*(a[0] + a[1] + a[2] - a[3] + a[4] - a[5]),
                   -(1/2)*(a[0] + a[1] + a[2] - a[3] - a[4] + a[5]),
                   -(1/2)*(a[0] + a[1] + a[2] - a[3] - a[4] - a[5]),
                   -(1/2)*(a[0] + a[1] - a[2] + a[3] + a[4] + a[5]),
                   -(1/2)*(a[0] + a[1] - a[2] + a[3] + a[4] - a[5]),
                   -(1/2)*(a[0] + a[1] - a[2] + a[3] - a[4] + a[5]),
                   -(1/2)*(a[0] + a[1] - a[2] + a[3] - a[4] - a[5]),
                   -(1/2)*(a[0] + a[1] - a[2] - a[3] + a[4] + a[5]),
                   -(1/2)*(a[0] + a[1] - a[2] - a[3] + a[4] - a[5]),
                   -(1/2)*(a[0] + a[1] - a[2] - a[3] - a[4] + a[5]),
                   -(1/2)*(a[0] + a[1] - a[2] - a[3] - a[4] - a[5]),
                   -(1/2)*(a[0] - a[1] + a[2] + a[3] + a[4] + a[5]),
                   -(1/2)*(a[0] - a[1] + a[2] + a[3] + a[4] - a[5]),
                   -(1/2)*(a[0] - a[1] + a[2] + a[3] - a[4] + a[5]),
                   -(1/2)*(a[0] - a[1] + a[2] + a[3] - a[4] - a[5]),
                   -(1/2)*(a[0] - a[1] + a[2] - a[3] + a[4] + a[5]),
                   -(1/2)*(a[0] - a[1] + a[2] - a[3] + a[4] - a[5]),
                   -(1/2)*(a[0] - a[1] + a[2] - a[3] - a[4] + a[5]),
                   -(1/2)*(a[0] - a[1] + a[2] - a[3] - a[4] - a[5]),
                   -(1/2)*(a[0] - a[1] - a[2] + a[3] + a[4] + a[5]),
                   -(1/2)*(a[0] - a[1] - a[2] + a[3] + a[4] - a[5]),
                   -(1/2)*(a[0] - a[1] - a[2] + a[3] - a[4] + a[5]),
                   -(1/2)*(a[0] - a[1] - a[2] + a[3] - a[4] - a[5]),
                   -(1/2)*(a[0] - a[1] - a[2] - a[3] + a[4] + a[5]),
                   -(1/2)*(a[0] - a[1] - a[2] - a[3] + a[4] - a[5]),
                   -(1/2)*(a[0] - a[1] - a[2] - a[3] - a[4] + a[5]),
                   -(1/2)*(a[0] - a[1] - a[2] - a[3] - a[4] - a[5]),
                  ])).like(DATA)
out = [None] * K
for n in range(0,K):
    addrzero = Fxp(0).like(ADDRtype)
    ACC = LUT[addrzero.val]
    ACC.rounding = 'trunc'

    for i in reversed(range(0,N)): # From LSB
        # Sign bit only for last bit
        Ts = 1 if 0==i else 0
    
        addr_str = "0b0" # Fxp ignores unsigned type while convert str to number, 1 in MSB cause an error
        for j in range(1,K):
            addr_str += xf[j].bin()[i]
        xn1bit = int(xf[0].bin()[i])
    
        assert(xn1bit == 0 or xn1bit == 1)
    
        xn1 = ~addrzero if 1 == xn1bit else addrzero # Extend xn1nit to addr format string
    
        addr = Fxp(addr_str).like(ADDRtype) ^ xn1 # XOR with xn1
        sign = xn1bit ^ Ts
    
        LUTval = LUT[addr.val]
    
        if sign:
            ACC.equal(ACC - LUTval)
        else:
            ACC.equal(ACC + LUTval)
        
        if 0!=i: ACC.equal(ACC >> 1) # Shift until last iteration
    
        print("|LUT:"+ str(float(LUTval)) +"|"+addr_str + "|" + xf[0].bin()[i]
              +"| xor:"+ addr.bin()+ "|"+ str(Ts) + "| sign:" + str(sign))
    
    out[n] = ACC
    print(n)
    for j in range(0,K):
        print(xf[j].bin())

|LUT:0.47998046875|0b000000|1| xor:11111|0| sign:1
|LUT:0.47998046875|0b000000|1| xor:11111|0| sign:1
|LUT:0.47998046875|0b000000|1| xor:11111|0| sign:1
|LUT:0.47998046875|0b000000|1| xor:11111|0| sign:1
|LUT:0.47998046875|0b000000|1| xor:11111|0| sign:1
|LUT:0.47998046875|0b000000|1| xor:11111|0| sign:1
|LUT:0.47998046875|0b000000|1| xor:11111|0| sign:1
|LUT:0.47998046875|0b000000|1| xor:11111|0| sign:1
|LUT:0.47998046875|0b000000|1| xor:11111|0| sign:1
|LUT:0.47998046875|0b000000|1| xor:11111|0| sign:1
|LUT:0.47998046875|0b000000|1| xor:11111|0| sign:1
|LUT:-0.5|0b000000|0| xor:00000|1| sign:1
0
011111111111
000000000000
000000000000
000000000000
000000000000
000000000000
|LUT:0.47998046875|0b000000|1| xor:11111|0| sign:1
|LUT:0.47998046875|0b000000|1| xor:11111|0| sign:1
|LUT:0.47998046875|0b000000|1| xor:11111|0| sign:1
|LUT:0.47998046875|0b000000|1| xor:11111|0| sign:1
|LUT:0.47998046875|0b000000|1| xor:11111|0| sign:1
|LUT:0.47998046875|0b000000|1| xor:11111|0| sign:1
|LUT:0.4799

In [17]:
for i in range(0,K):
    print(float(out[i]))

0.02001953125
0.02001953125
0.02001953125
0.02001953125
0.02001953125
0.02001953125
